# 01 - Experiment Objective

This experiment evaluates membership leakage in the models produced by the utility evaluation experiment.

A shadow-model-based Membership Inference Attack is used to determine whether an attacker can distinguish training members from non-members based on black-box model behavior.

The experiment consumes the artifact generated by the utility evaluation pipeline and evaluates leakage across all available datasets, tasks, targets, and models.

# 02 - Experimental Definition

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.core.attacks_config import ShadowAttackConfig
from src.core.models_spec_config import ModelSpec

In [ ]:
attack_shadow_model_config = ModelSpec(
    name="xgboost",
    model_type="xgboost_classifier",
    parameters={
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": 42,
        "verbosity": 0,
    },
)

shadow_attack_config = ShadowAttackConfig(
    enabled=True,
    n_shadow_models=3,
    member_fraction=0.5,
    attack_test_size=0.3,
    attack_features=[
        "probabilities",
        "confidence",
        "entropy",
        "loss",
    ],
    attack_model=attack_shadow_model_config,
)

SEED = 42
ATTACK_TYPE = "shadow_model_membership_inference"

# 03 - Load Utility Artifact

In [ ]:
from artifacts.persistence import load_utility_leakage_input

UTILITY_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "evaluation"
UTILITY_ARTIFACT_PATH = max(
    UTILITY_ARTIFACT_ROOT.glob("evaluation_*"),
    key=lambda path: path.name,
)

UTILITY_ARTIFACT_PATH

In [ ]:
leakage_input, metadata = load_utility_leakage_input( UTILITY_ARTIFACT_PATH )

source_experiment_id = metadata["experiment_id"]

print(f"Source utility experiment: {source_experiment_id}")
print(f"Leakage records: {len(leakage_input)}")

# 04 - Prepare Attack Inputs

In [ ]:
REQUIRED_ATTACK_INPUT_FIELDS = [
    "dataset",
    "task_type",
    "target",
    "model",
    "model_type",
    "X_pool",
    "y_pool",
]

attack_inputs = leakage_input.to_dict("records")

for index, attack_input in enumerate(attack_inputs):
    missing_fields = [
        field
        for field in REQUIRED_ATTACK_INPUT_FIELDS
        if field not in attack_input
    ]

    if missing_fields:
        raise ValueError(
            f"Attack input {index} is missing required fields: {missing_fields}"
        )

    if attack_input["task_type"] not in {"classification", "regression"}:
        raise ValueError(
            f"Unsupported task_type in attack input {index}: {attack_input['task_type']}"
        )

    if "target_prediction" not in attack_input or attack_input["target_prediction"] is None:
        raise ValueError(
            "Shadow MIA requires target_prediction in leakage_input.pkl for "
            f"attack input {index}. Re-run the Utility notebook with the current "
            "artifact schema if this field is absent."
        )

print(f"Attack records: {len(attack_inputs)}")

In [ ]:
model_specs = {
    (model["name"], model["model_type"]): ModelSpec(
        name=model["name"],
        model_type=model["model_type"],
        parameters=model.get("parameters", {}),
    )
    for model in metadata.get("models", [])
}

missing_model_specs = [
    (attack_input["model"], attack_input["model_type"])
    for attack_input in attack_inputs
    if (attack_input["model"], attack_input["model_type"]) not in model_specs
]

if missing_model_specs:
    raise ValueError(
        "Utility metadata does not contain model specifications for: "
        f"{sorted(set(missing_model_specs))}"
    )

# 05 - Execute Shadow MIA

In [ ]:
from src.experiments.utility_evaluation_services.model import model_runner
from src.experiments.lekage_evaluation.attacks.run_shadow_mia import (
    run_shadow_model_membership_inference_attack,
)


def build_shadow_model_runner(model_spec):
    def execute_shadow_model(prepared_dataset, task_type, seed):
        parameters = dict(model_spec.parameters)

        if "random_state" in parameters:
            parameters["random_state"] = seed

        seeded_model_spec = ModelSpec(
            name=model_spec.name,
            model_type=model_spec.model_type,
            parameters=parameters,
        )

        return model_runner.execute_model(
            prepared_features=prepared_dataset,
            model_spec=seeded_model_spec,
        )

    return execute_shadow_model

In [ ]:
attack_results = []

for index, attack_input in enumerate(attack_inputs):
    model_spec = model_specs[
        (attack_input["model"], attack_input["model_type"])
    ]

    result = run_shadow_model_membership_inference_attack(
        model_runner=build_shadow_model_runner(model_spec),
        task_type=attack_input["task_type"],
        X_pool=attack_input["X_pool"],
        y_pool=attack_input["y_pool"],
        target_prediction_result=attack_input["target_prediction"],
        shadow_config=shadow_attack_config,
        seed=SEED + index,
    )

    attack_results.append(
        {
            "input": attack_input,
            "result": result,
        }
    )

print(f"Attack executions: {len(attack_results)}")

# 06 - Evaluate Attack

In [ ]:
from src.experiments.lekage_evaluation.metrics import compute_attack_metrics

for attack_result in attack_results:
    result = attack_result["result"]

    target_metrics = compute_attack_metrics(
        y_true=result["target_labels"],
        y_pred=result["target_predictions"],
    )

    shadow_validation_metrics = compute_attack_metrics(
        y_true=result["shadow_validation_labels"],
        y_pred=result["shadow_validation_predictions"],
    )

    attack_result["metrics"] = target_metrics
    attack_result["shadow_val_acc"] = shadow_validation_metrics.attack_acc

# 07 - Build Results

In [ ]:
import pandas as pd
from datetime import datetime
from dataclasses import asdict

experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")

attack_records = []

for attack_result in attack_results:
    attack_input = attack_result["input"]

    record = {
        "experiment_id": experiment_id,
        "source_experiment_id": source_experiment_id,
        "dataset": attack_input["dataset"],
        "task_type": attack_input["task_type"],
        "target": attack_input["target"],
        "model": attack_input["model"],
        "model_type": attack_input["model_type"],
        "attack_type": ATTACK_TYPE,
        "n_shadow_models": shadow_attack_config.n_shadow_models,
        "member_fraction": shadow_attack_config.member_fraction,
        "attack_test_size": shadow_attack_config.attack_test_size,
        "attack_features": ",".join(shadow_attack_config.attack_features),
        "attack_model": shadow_attack_config.attack_model.name,
        "attack_model_type": shadow_attack_config.attack_model.model_type,
        "shadow_val_acc": attack_result["shadow_val_acc"],
    }

    record.update(asdict(attack_result["metrics"]))
    attack_records.append(record)

attack_metrics = pd.DataFrame(attack_records)
attack_metrics

# 08 - Persist MIA Artifact

In [ ]:
from artifacts.persistence import persist_membership_attack_artifact

EXPERIMENT_TYPE = "membership_inference"
ARTIFACT_SCHEMA_VERSION = "1.0"

experiment_metadata = {
    "experiment_id": experiment_id,
    "experiment_type": EXPERIMENT_TYPE,
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_artifact": {
        "experiment_type": "utility_evaluation",
        "experiment_id": source_experiment_id,
        "path": str(UTILITY_ARTIFACT_PATH),
    },
    "attack": {
        "type": ATTACK_TYPE,
        "n_shadow_models": shadow_attack_config.n_shadow_models,
        "member_fraction": shadow_attack_config.member_fraction,
        "attack_test_size": shadow_attack_config.attack_test_size,
        "attack_features": shadow_attack_config.attack_features,
        "attack_model": {
            "name": shadow_attack_config.attack_model.name,
            "model_type": shadow_attack_config.attack_model.model_type,
            "parameters": shadow_attack_config.attack_model.parameters,
        },
    },
    "records": {
        "attack_input_count": len(attack_inputs),
        "attack_result_count": len(attack_results),
    },
}

artifact_path = persist_membership_attack_artifact(
    dir_path= UTILITY_ARTIFACT_PATH,
    metadata=experiment_metadata,
    attack_metrics=attack_metrics,
    attack_results=attack_results,
)

artifact_path

# 09 - Inspect Results

In [ ]:
attack_metrics.head()

In [ ]:
attack_metrics.shape

In [ ]:
attack_metrics.groupby(
    ["dataset", "task_type", "model", "attack_type"],
    as_index=False,
).agg(
    attack_records=("experiment_id", "count"),
    mean_attack_acc=("attack_acc", "mean"),
    mean_advantage=("advantage", "mean"),
)

# 10 - Conclusion

This notebook attacked all records available in the Utility leakage artifact.

The processed datasets, tasks, and models are represented in `attack_metrics` and summarized in the inspection cells above.

The Shadow MIA configuration is centralized in `shadow_attack_config`.

The MIA artifact was persisted to `artifact_path`. No result interpretation is performed in this notebook.